# AI-Overview
- Basic things that needed a refresh

## Search Problems
- consists of:
  - State space
  - successor function
  - Start state and a goal test
    - *goal test* - tests if goal has been achieved
- *Solution* - sequence of actions (a plan) which transforms teh start state to a goal state
-
### Traveling in Romania example:
- State Space:
  - Cities
- Successor functions:
  - Roads: Go to adjacent city with cost = distance
- Start State:
  - Arad (arbitarty village)
- Goal Test:
  - Is state == Bucharest
- Solution
  - Finding the path

### State Space Graph vs Search Trees
- State Space Graph: can be circular
- Search Tree: Could go on infinitley
- Both represent the same thing

## General Tree Seach Formula
```
*returns a solution or failure*
initialize the search tree using the initial state of the problem
loop do:
  - if there are no canidates for expansion, return failure
  - Choose a leaf node for expansion according to strategy
  - if node containns a goal state, then return the corresponding solution
    - else, expand the node and add the resulting nodes to the search tree
end
```
- DFS, BFS, etc all fall under this umbrella
- UCS: picks the node with the lowest path cost from the start node so far
- A* search
  - PQ ordered by Sum of UCS and Greedy
    - Greedy: forward cost
- *completeness* - guaranteed to find a solution when one exists
- *optimality* - Whenever it finds a solution, it is guaranteed to be the best possible one.
## Hueristics
- a function that estimates how close a state is to a goal
- designed on a problem to problem basis
- ex: manhattan distance, euclidean distance for pathing
### Admissibility
- Inadmissibile - pessimistic
- Admissible - optimistic
- One is admisible if the estimate is >= 0 and <= the actual distance to the goal
### Consisteny
- aka monotonic
- h(n) <= cost(n->n') + h(n')
  - must hold for every node `n` and every successor of `n`, `n'`
## Tree Search vs Graph Search
- Tree search is extra work as it fails to detect repeated states which can cause exponentially more work.
- Thus, we use graph search which accounts for repeated nodes
### Graph Search Explanation
- Main idea: never expand a state twice
- implemented on HW 1, basically just remember what states you have visited with a state set
  - vital to store this as a set and not a list (memory)
### Optimality of certain search algs
- Tree Search:
  - A* is optimal if hueristic is admissible
  - UCS is a special case (h=0)
- Graph Search:
  - A* is optimal if hueristic is consistent
  - UCS optimal (h=0 is consisten)
- **Consistency implies Admissibility**
- In general, most natural admissible heuristics tend to be consistent, especially if from relaxed problems.

## Constraint Satisfaction Problems
- Different from standard search Problems where:
  - State is a 'black box': arbitrary data structure
  - Goal test can be any function over states
  - Successor function can also be anything

- CSPs:
  - a special subset of search problems
  - Where state is defined by variables $X_i$ with values from a Domain D (sometimes D depends on i)
  - Goal test is a set of constraints specifying allowable combinations of values for subsets of variables


### Example
![example](https://www.cs.cmu.edu/~15281-s23/coursenotes/constraints/images/australia.png)

- Variables: WA, NT, Q, NSQ, V, SA, T
- Domains: D = {red, green, blue}
- Constraints:
  - adjacent regions must have different colors.
    - Implicit example (general rule that must be satisfied):
      - WA != NT
    - Explicit example (list of all allowed combinations):
      - (WA, NT) ∈ { (red, green), (red, blue), (green, red), (green, blue), (blue, red), (blue, green), ... }
- A solution would be assignments satisfying all constrains such as:
  - {WA=red, NT=greex, Q=red, NSW=green, V=red, SA=blue, T=green}
### Constaint graphs
- Show how each region are related, in this case maps which are adjacent
- *Binary CSP* - each constrain relates (at most) two variables.
- *Binary constraint graph*: nodes are variables, arcs show constraints
- General purpose CSP algs use the graph structure to speed up search
### Varieties Of Constrains
- *Unary constraints* - involve a single variable
  - Ex: SA != green
- *Binary Constraints* - involve pairs of variables
  - Ex: SA != WA
- Higher-order constraints involve 3 or more variables
  - e.g. cryptarithmetic column constraints
- *Soft Constraints*
  - aka preferences
  - often representable by a cost for each variable assignment
  - gives constrained optimization problem
  - Ignored until hit bayes' nets
  - Ex: red is better than green

## Solving CSP's

### Backtracking Search
- basic uniformed alg for solving CSPs
- Idea 1: One variable at a time
  - variable assignments are commutative, so fix ordering because doing so will NOT affect the final result`.
  - Only need to consider assignments to a single variable at each step
    - simply peice those together to form a solution, eventually, after doing some backtracking and shuffling, you will find one
- Idea 2: check constraints as you go
  - ex: consider only values which do not conflict with previous assignments.
  - Might have to do some computation to check the constraints.
  - 'incremental goal test'
- Backtracking = DFS + variable-ordering + fail-on-violation
- Improving Backtracking
  - General-purpose ideas give huge gains in speed and are more widely applicable that hueristics
- Ordering:
  - Which variable should be assigned next?
  - in what order should its values be tried?
- *Filtering*: Can we detect inevitable failure early?
- *Structure*: Can we exploit the problems structure

### Filtering (Forward Checking)
- *Filtering* - Keep track of domains for unassigned variables and cross off bad options.
- *Forward Checking* - Cross off values that violate a constraint when added to the existing assignment.
  - enforcing consistency of arcs pointing to each new assignment. But other types of arc consistency algorithms are possible.
- Foward checking propagates information from assigned to unassigned variables, but doesn't provide early detection for all failures.
- Consistency of a single arc:
  - arc X->Y is consistent iff for every x in the tail, there is some y in the head which could be assigned without violating a constraint.

#### Arc Consistency of an Entire CSP
- A simple form of propagation makes sure all arcs are consistent:
  - Remember to delete from the tail
- If X changes, neighbors of X need to be rechecked
- This format detects failure earlier than forward checking
- Can be run as a preprocessor or after each assignment
- Limitations:
  - after encorcing arch consistency:
    - Can have one solution left
    - Can have multiple solutions left
    - Can have no solutions left (**and not know it**)
- Arc consistency still runs inside a backtracking search!

### Ordering
- Which var shouldbe assigned next.
- Minimum Remaining Values (MRV) is a ordering framework
  - it involves choosing the variable with the fewest legal left values in it's domain.
    - This variable is aka "most constrained variable"
  - aka fail fast ordering
- Least Constraining Value (LCV)
  - Given a choice of variable, choose the least constraining value
    - i.e the one that rules out the fewest values in the remaining variables.
      - it could take computation to determine this (like rerunning forward checking)
### Structure
#### Tree-Structured CSP's
- *Theorem* - if the constraint graph has no loops, the CSP can be solved in O(n $d^2$) time
  - In comparison to general CSPs where the worst-case time is O($d^n$)
  - *n* - number of variables to assign in the CSP
  - *d* - Maximum number of possible values any single variable can take (Domain Size)
- Algorithm for tree-structured CSPs:
  - Order: choose a root variable, order variables so that the parents precede children
  - Remove Backward:
    - Start at i = n (furthest Child)
    - Decrement i all the way down in i = 2
    -  in between each decrement, perform RemoveInconsistentFromPossDomain(Parent($X_i$), $X_i$)
  - Assign Forward:
    - for i = 1 -> n
      - assign $X_i$ (domain option) consistently with Parent($X_i$)
- Claims about Tree-Structured CSPs
  - After backward pass, all root-to-leaf arcs are consistent
    - Proof: each X->Y was made consistent at one point and Y's domain could not have been reduced thereafter as Y's children were processed before Y
  - If root-to-leaf arcs are consistent, forward assignment will not backrack (undo assignment and replace with a different value)
    - This does not work with cycles in the constraint graphs
      - because if two arcs point to the same node, they are guaranteed to both have individually consistent assignments - but those assignments might conflict.
### Improving Structure
- Creating nearly Tree-Structured CSPs using cutsets
- *conditioning* - instantiate a variable, prune its neighbors' domains
- *Cutset Conditioning* - instantiate (in all ways) a set of variables specifically so that you can cut them out and turn the graph into a tree
- Have to instantiate the cuset in all possible ways so that you can identify and correctly find all solutions.
  - you create a tree of possible permutations with the rest of the nodes having reduced domains and evaluate possible solutions from there
- Cutset size c gives runtime $O((d^c)(n-c)d^2)$
  - very fast for small c

## Types of Tasks (Games)
- We want algorithms for calculating a strategy (policy) which recommends a move from each state.
### Deterministic Games
- Many formalizations, one being:
  - States
  - Players
  - Actioins
  - Transitions Function
    - (State change)
  - Terminal Test
  - Terminal Utilities
- Solution for a player would be a policy: S->A
- recall *deterministic* - identical values will ALWAYS produce idential output.

### Zero-Sum Games
- Agents have opposite utilities
- lets us think of a single value that one maximizes and the other minimizes
- adversarial, pure competition
- As compared to **General Games**
  - Agents have independent utilities (values on outcomes)
  - Cooperations, indifference, competition and more are possible as opposed too just competition

### Adversarial Search
- most effective for zero sum games
- Minmax search:
  - tree in which players alternate at each layer to either min or max the values, working up the tree
  - A state-space search tree
  - players alternate turns
  - *Terminal State* - state where the game has ended, no more moves can be made
  - *Down arrow* - find min of children
  - *Up arrow* - find max of children

### Game tree Pruning (Specifically Minimax)
-  Way of minimizing resource toll via cuting out ndoes that will not affect final result

## Rources Limits

### Evaluation Funcion
- these functions score non-terminal states by using depth-limited search
  - *depth-limited* - have a hyperparam that restrics how deep the alg or process will go.
- The ideal function is to return the actual minimax value of the position
  - in practice this is typically a eighted linear sum of features
  - Example: $f_1(s)$ = (num white queens - num black queens)
- uncertain outcomes controlled by chance, not an adversary

### Expectimax Search
- Values associated with states now reflect average-case (expectimax) outcomes, not worst-case (minimax) outcomes.
- Objective: compute average score under optimal play
  - has max nodes as minimax search
  - *chance nodes* - like min nodes but the outcome is uncertain
    - calculate the *expected utilities* of the nodes by taking the weighted average (expectation) of the children.

### Modeling Assumptions
- *Dangerous Optimisim* - assuming chance when the world is adversarial
- *Dangerous Pessimism* - assuming the worst case when it's not likely
- In a pacman model:
  - *Minimax Pacman* assumes ghost is adversarial
  - *Expectimax Pacman* - assumes ghost moves randomly
- **Using Minimax against a random opponent is not terrible and often salvageable, while using Expectimax against an adversarial opponent is disastrous**

### Utilities
